In [4]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

In [2]:
def summarize_text(file_path: str) -> str:
    # TXT 파일 읽기
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    # 시스템 프롬프트
    system_prompt = f"""
    너는 다음 글을 요약하는 봇이다.
    아래 글을 읽고, 저자의 문제 인식과 주장을 파악하고 주요 내용을 요약하라.

    작성해야 하는 포맷은 다음과 같다.

    # 제목

    ## 저자의 문제 인식 및 주장 (15문장 이내)

    ## 저자 소개

    ## 주요 내용

    --------------- 이하 텍스트 ---------------

    {text}
    """

    print(system_prompt)
    print("=========================================")

    # OpenAI API 호출
    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            }
        ],
    )

    return response.choices[0].message.content


if __name__ == "__main__":

    file_path = (
        "output/From Passive Retrieval to Active Memory Navigation Learning to Use Memory as a Structured Action Space_with_preprocessing.txt"
    )

    # 요약 실행
    summary = summarize_text(file_path)

    print(summary)

    # 요약 결과 저장
    output_path = "output/From Passive Retrieval to Active Memory Navigation Learning to Use Memory as a Structured Action Space_with_preprocessing_summary.txt"

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(summary)

    print(f"\n요약 결과 저장 완료: {output_path}")


    너는 다음 글을 요약하는 봇이다.
    아래 글을 읽고, 저자의 문제 인식과 주장을 파악하고 주요 내용을 요약하라.

    작성해야 하는 포맷은 다음과 같다.

    # 제목

    ## 저자의 문제 인식 및 주장 (15문장 이내)

    ## 저자 소개

    ## 주요 내용

    --------------- 이하 텍스트 ---------------

    From Passive Retrieval to Active Memory
Navigation: Learning to Use Memory as a
Structured Action Space
Yue Xu*1,2, Yutao Sun*3, Yihao Liu*4, Mengyu Zhou†1, Jiayi Qiao*5, Lu Ma1, Kai Tang1, Wenjie Wang†2, Xiaoxi
Jiang1 and Guanjun Jiang1
1Qwen Large Model Application Team, Alibaba, 2ShanghaiTech University, 3Zhejiang University, 4Peking University, 5National
University of Singapore
*Work done during an internship at Alibaba. †Corresponding author.
Long-term user memory is essential for personalized conversational agents, yet many memory systems
still expose memory through passive retrieval interfaces, making the model a consumer of pre-selected
evidence. We introduce NapMem, a framework for learning to use long-term user memory as a structured
action space rather than passi

## 최종본
- input으로 파일 경로를 받도록 개선

In [5]:
import pymupdf
import os

In [6]:
# ============================================================
# 1. PDF 파일 경로 입력
# ============================================================

while True:

    pdf_file_path = input(
        "요약할 PDF 경로를 입력하세요: "
    ).strip()

    if not pdf_file_path:
        print("❌ 파일 경로를 입력해주세요.\n")
        continue

    if not os.path.isfile(pdf_file_path):
        print(
            f"❌ 파일을 찾을 수 없습니다.\n"
            f"입력한 경로: {pdf_file_path}\n"
        )
        continue

    if not pdf_file_path.lower().endswith(".pdf"):
        print("❌ PDF 파일만 입력해주세요.\n")
        continue

    break


# ============================================================
# 2. PDF → TXT 전처리
# ============================================================

print("\n[1/3] PDF를 읽는 중입니다...")

doc = pymupdf.open(pdf_file_path)

header_height = 80
footer_height = 80

full_text = ""

for page in doc:

    rect = page.rect

    # 헤더와 푸터를 제외한 본문만 추출
    text = page.get_text(
        clip=(
            0,
            header_height,
            rect.width,
            rect.height - footer_height
        )
    )

    full_text += (
        text
        + "\n------------------------------------\n"
    )

doc.close()


# ============================================================
# 3. 전처리 TXT 저장
# ============================================================

os.makedirs("output", exist_ok=True)

pdf_file_name = os.path.splitext(
    os.path.basename(pdf_file_path)
)[0]

txt_file_path = os.path.join(
    "output",
    f"{pdf_file_name}_with_preprocessing.txt"
)

with open(
    txt_file_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(full_text)

print(
    f"✓ 전처리 완료\n"
    f"  저장: {txt_file_path}"
)


# ============================================================
# 4. TXT 파일 읽기
# ============================================================

print("\n[2/3] 문서를 읽는 중입니다...")

with open(
    txt_file_path,
    "r",
    encoding="utf-8"
) as f:
    text = f.read()


# ============================================================
# 5. 요약 프롬프트
# ============================================================

system_prompt = f"""
너는 다음 글을 요약하는 봇이다.

아래 글을 읽고,
저자의 문제 인식과 주장을 파악하고,
주요 내용을 요약하라.

작성해야 하는 포맷은 다음과 같다.

# 제목

## 저자의 문제 인식 및 주장 (15문장 이내)

## 저자 소개

## 주요 내용

--------------- 이하 텍스트 ---------------

{text}
"""


# ============================================================
# 6. OpenAI API 호출
# ============================================================

print("\n[3/3] OpenAI에게 문서 요약을 요청하는 중입니다...")

response = client.chat.completions.create(
    model="gpt-4o",
    temperature=0.1,
    messages=[
        {
            "role": "system",
            "content": system_prompt
        }
    ]
)

summary = response.choices[0].message.content


# ============================================================
# 7. 요약 결과 저장
# ============================================================

summary_file_path = os.path.join(
    "output",
    f"{pdf_file_name}_summary.txt"
)

with open(
    summary_file_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(summary)


# ============================================================
# 8. 결과 출력
# ============================================================

print(
    "\n=================================================="
)

print("요약 완료!")

print(
    "=================================================="
)

print(f"\n원본 PDF:")
print(f"  {pdf_file_path}")

print(f"\n전처리 TXT:")
print(f"  {txt_file_path}")

print(f"\n요약 TXT:")
print(f"  {summary_file_path}")

print(
    "\n================ 요약 결과 =================\n"
)

print(summary)


[1/3] PDF를 읽는 중입니다...
✓ 전처리 완료
  저장: output\Mayer2026SSTracing_with_preprocessing.txt

[2/3] 문서를 읽는 중입니다...

[3/3] OpenAI에게 문서 요약을 요청하는 중입니다...

요약 완료!

원본 PDF:
  data/Mayer2026SSTracing.pdf

전처리 TXT:
  output\Mayer2026SSTracing_with_preprocessing.txt

요약 TXT:
  output\Mayer2026SSTracing_summary.txt

================ 요약 결과 =================

# Ultrafast Screen Space Refractions and Caustics via Newton's Method

## 저자의 문제 인식 및 주장 (15문장 이내)
저자들은 투명한 물질의 굴절과 카우스틱 효과를 실시간으로 렌더링하는 데 있어 기존의 레이 마칭 방식이 느리고 비효율적이라는 문제를 인식했다. 이 문제를 해결하기 위해 뉴턴의 방법을 사용하여 스크린 공간에서 굴절을 처리하는 새로운 알고리즘을 제안한다. 이 방법은 각 픽셀에 대해 굴절된 광선의 깊이와 G-버퍼 깊이의 차이를 함수로 정의하고, 뉴턴의 방법을 통해 빠르게 수렴하여 픽셀당 비용을 줄인다. 또한, 깊이 불연속성이나 초기 추정치의 부정확성 같은 일반적인 실패 사례를 분석하고 이를 감지하고 해결할 수 있는 실용적인 전략을 제안한다. 이 방법은 특히 두꺼운 투명 물체나 경사각에서의 굴절을 처리하는 데 있어 기존 방법보다 빠른 수렴을 보여준다. 또한, 이 방법을 사용하여 스크린 공간에서의 카우스틱 효과를 가속화할 수 있음을 입증한다. 저자들은 이 방법이 실시간 응용 프로그램에서의 엄격한 프레임 시간 예산을 충족할 수 있는 예측 가능한 성능을 제공한다고 주장한다. 실패 사례가 발생하더라도 표준 스크린 공간 방법을 대체 방법으로 사용하여 시각적 오류를 완화할 수 있다. 이 방법은 스크린